[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session3/solutions/Assignment_Pharmacy_Pipeline_Solutions.ipynb)

# Session 3 Assignment: The Complete Pharmacy Intelligence Pipeline — SOLUTIONS
## CCI — Clinical AI Development

### Grading
| Part | Weight | Description |
|------|--------|-------------|
| Part 1 | 25% | CSV to SQL — Load and query pharmacy data |
| Part 2 | 25% | Structured Output — Extract drug info from clinical notes |
| Part 3 | 25% | Text-to-SQL — Pharmacist natural language interface |
| Part 4 | 15% | Critical Analysis — Clinical risks and limitations |
| Part 5 | 10% | Stretch Challenge — Tool calling OR benchmarking |

---
## Setup

In [ ]:
# Setup — clone the repo and install packages
import os

# if not os.path.exists('/content/CCI'):
#     !git clone https://github.com/IyadSultan/CCI.git /content/CCI
# os.chdir('/content/CCI/session3/solutions')

# !pip install -q openai pydantic pandas

import json
import pandas as pd
import sqlite3
from openai import OpenAI
from pydantic import BaseModel
from typing import List, Optional
# from google.colab import userdata

# Make sure your API key is stored in Colab Secrets (click the key icon in the left sidebar)
# under the name: OPENAI_API_KEY
api_key = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

print(f"Working directory: {os.getcwd()}")
print("Setup complete!")

ModuleNotFoundError: No module named 'google.colab'

---
# Part 1: CSV to SQL — Load and Query Pharmacy Data (25%)

### Cell 1.1 — Load CSVs into SQLite

In [ ]:
# === CELL 1.1: LOAD DATA INTO SQLITE ===

df_pharmacy = pd.read_csv('../data/vista_pharmacy.csv')
df_patients = pd.read_csv('../data/vista_patients.csv')

conn = sqlite3.connect('khcc_pharmacy.db')
df_pharmacy.to_sql('vista_pharmacy', conn, if_exists='replace', index=False)
df_patients.to_sql('vista_patients', conn, if_exists='replace', index=False)

result_pharm = pd.read_sql("SELECT COUNT(*) as total FROM vista_pharmacy", conn)
result_pts = pd.read_sql("SELECT COUNT(*) as total FROM vista_patients", conn)
print(f"Database ready: {result_pharm['total'][0]} pharmacy orders, {result_pts['total'][0]} patients")

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f"Tables: {list(tables['name'])}")

### Cell 1.2 — Explore Pharmacy Data

In [ ]:
# === CELL 1.2: EXPLORE PHARMACY DATA ===

print(f"Shape: {df_pharmacy.shape}")
print(f"\nUnique patients: {df_pharmacy['MRN'].nunique()}")

print(f"\nTop 10 drugs:")
print(df_pharmacy['DRUG'].value_counts().head(10))

print(f"\nStatus distribution:")
print(df_pharmacy['STATUS'].value_counts())

print(f"\nRoute distribution:")
print(df_pharmacy['ROUTE'].value_counts())

print(f"\nDate range:")
print(f"  Earliest: {df_pharmacy['ISSUE_DATE'].min()}")
print(f"  Latest:   {df_pharmacy['ISSUE_DATE'].max()}")

### Cell 1.3 — SQL Queries

In [ ]:
# === CELL 1.3: SQL QUERIES ===

# Query 1: Top 10 prescribed drugs
print("=== Query 1: Top 10 Prescribed Drugs ===")
q1 = pd.read_sql("""
    SELECT DRUG, COUNT(*) as order_count
    FROM vista_pharmacy
    GROUP BY DRUG
    ORDER BY order_count DESC
    LIMIT 10
""", conn)
print(q1)

# Query 2: Prescriptions per clinic
print("\n=== Query 2: Prescriptions Per Clinic ===")
q2 = pd.read_sql("""
    SELECT CLINIC, COUNT(*) as order_count
    FROM vista_pharmacy
    GROUP BY CLINIC
    ORDER BY order_count DESC
""", conn)
print(q2)

# Query 3: Count by STATUS
print("\n=== Query 3: Orders by Status ===")
q3 = pd.read_sql("""
    SELECT STATUS, COUNT(*) as order_count
    FROM vista_pharmacy
    GROUP BY STATUS
    ORDER BY order_count DESC
""", conn)
print(q3)

# Query 4: Average days supply for top 5 drugs
print("\n=== Query 4: Average Days Supply (Top 5 Drugs) ===")
q4 = pd.read_sql("""
    SELECT DRUG, ROUND(AVG(DAYS_SUPPLY), 1) as avg_days, COUNT(*) as orders
    FROM vista_pharmacy
    GROUP BY DRUG
    ORDER BY orders DESC
    LIMIT 5
""", conn)
print(q4)

# Query 5: Patients with refills > 0
print("\n=== Query 5: Patients with Refills ===")
q5 = pd.read_sql("""
    SELECT MRN, DRUG, Number_OF_REFILLS
    FROM vista_pharmacy
    WHERE Number_OF_REFILLS > 0
    ORDER BY Number_OF_REFILLS DESC
    LIMIT 15
""", conn)
print(q5)

### Cell 1.4 — JOIN Queries

In [ ]:
# === CELL 1.4: JOIN QUERIES ===

# JOIN Query 1: Prescriptions by nationality
print("=== JOIN 1: Prescriptions by Nationality ===")
jq1 = pd.read_sql("""
    SELECT p.NATIONALITY, COUNT(*) as rx_count, COUNT(DISTINCT rx.MRN) as unique_patients
    FROM vista_pharmacy rx
    JOIN vista_patients p ON rx.MRN = p.MRN
    GROUP BY p.NATIONALITY
    ORDER BY rx_count DESC
""", conn)
print(jq1)

# JOIN Query 2: Drugs for patients over 60
print("\n=== JOIN 2: Drugs for Patients Over 60 ===")
jq2 = pd.read_sql("""
    SELECT DISTINCT rx.DRUG, COUNT(*) as times_prescribed
    FROM vista_pharmacy rx
    JOIN vista_patients p ON rx.MRN = p.MRN
    WHERE (julianday('now') - julianday(p.DOB)) / 365.25 > 60
    GROUP BY rx.DRUG
    ORDER BY times_prescribed DESC
    LIMIT 10
""", conn)
print(jq2)

# JOIN Query 3: Male vs female prescription counts
print("\n=== JOIN 3: Male vs Female Prescription Counts ===")
jq3 = pd.read_sql("""
    SELECT p.SEX, COUNT(*) as rx_count, COUNT(DISTINCT rx.MRN) as unique_patients
    FROM vista_pharmacy rx
    JOIN vista_patients p ON rx.MRN = p.MRN
    GROUP BY p.SEX
""", conn)
print(jq3)

---
# Part 2: Structured Output — Extract Drug Info from Clinical Notes (25%)

### Cell 2.1 — Define Pydantic Models

In [ ]:
# === CELL 2.1: PYDANTIC MODELS ===

class DrugMention(BaseModel):
    name: str
    dose: Optional[str] = None
    frequency: Optional[str] = None
    route: Optional[str] = None

class DrugInteractionAlert(BaseModel):
    drug1: str
    drug2: str
    risk_level: str
    description: str

class NoteExtraction(BaseModel):
    patient_name: str
    mrn: str
    medications: List[DrugMention]
    allergies: List[str]
    potential_interactions: List[DrugInteractionAlert]
    key_findings: List[str]

print("Pydantic models defined.")
print(f"NoteExtraction fields: {list(NoteExtraction.model_fields.keys())}")

### Cell 2.2 — Load Clinical Notes

In [ ]:
# === CELL 2.2: LOAD CLINICAL NOTES ===

with open('../data/synthetic_clinical_notes.json', 'r') as f:
    notes = json.load(f)

print(f"Loaded {len(notes)} clinical notes.")
print(f"\nFirst note:")
print(f"  ID: {notes[0]['note_id']}")
print(f"  Type: {notes[0]['note_type']}")
print(f"  Text (first 200 chars): {notes[0]['text'][:200]}...")

### Cell 2.3 — Extract Structured Data from All 10 Notes

In [ ]:
# === CELL 2.3: EXTRACT FROM ALL 10 NOTES ===

extractions = []

for i, note_data in enumerate(notes):
    print(f"\n{'='*60}")
    print(f"Processing {note_data['note_id']} ({note_data['note_type']})")

    try:
        result = client.beta.chat.completions.parse(
            model="gpt-4o-mini",
            temperature=0.0,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a clinical pharmacist data extractor at KHCC. "
                        "Extract all medications (with doses and frequencies), allergies, "
                        "potential drug-drug interactions, and key clinical findings from "
                        "the oncology note. For interactions, assess risk level as high, "
                        "moderate, or low. Use 'NKDA' if no allergies are mentioned."
                    )
                },
                {"role": "user", "content": note_data['text']}
            ],
            response_format=NoteExtraction
        )

        parsed = result.choices[0].message.parsed
        extractions.append(parsed)

        print(f"  Patient: {parsed.patient_name} (MRN: {parsed.mrn})")
        print(f"  Medications: {len(parsed.medications)}")
        for med in parsed.medications:
            print(f"    - {med.name} {med.dose or ''} {med.frequency or ''}")
        print(f"  Allergies: {parsed.allergies}")
        print(f"  Interactions flagged: {len(parsed.potential_interactions)}")
        for interact in parsed.potential_interactions:
            print(f"    - [{interact.risk_level.upper()}] {interact.drug1} + {interact.drug2}: {interact.description}")

    except Exception as e:
        print(f"  ERROR: {e}")
        extractions.append(None)

print(f"\n\nExtraction complete: {sum(1 for e in extractions if e is not None)}/{len(notes)} notes processed.")

### Cell 2.4 — Cross-Reference with Pharmacy Database

In [ ]:
# === CELL 2.4: CROSS-REFERENCE ===

pharmacy_drugs = pd.read_sql(
    "SELECT DISTINCT UPPER(PHARMACY_ORDERABLE_ITEM) as drug_name FROM vista_pharmacy", conn
)
pharmacy_drug_set = set(pharmacy_drugs['drug_name'].tolist())
print(f"Unique drugs in pharmacy DB: {len(pharmacy_drug_set)}")
print(f"Drugs: {sorted(pharmacy_drug_set)}\n")

found_count = 0
not_found_count = 0

for ext in extractions:
    if ext is None:
        continue
    print(f"\n{ext.patient_name} ({ext.mrn}):")
    for med in ext.medications:
        med_upper = med.name.upper().split()[0]
        if med_upper in pharmacy_drug_set:
            print(f"  {med.name} — FOUND in pharmacy")
            found_count += 1
        else:
            print(f"  {med.name} — NOT FOUND in pharmacy")
            not_found_count += 1

print(f"\n--- Summary ---")
print(f"Drugs matched to pharmacy: {found_count}")
print(f"Drugs not in pharmacy: {not_found_count}")

### Cell 2.5 — Compare Against Ground Truth

In [ ]:
# === CELL 2.5: COMPARE AGAINST GROUND TRUTH ===

total_expected_meds = 0
total_extracted_meds = 0
total_expected_interactions = 0
total_extracted_interactions = 0
complete_notes = 0

for i, note_data in enumerate(notes):
    ext = extractions[i]
    if ext is None:
        continue

    expected = note_data['expected_extraction']

    # --- Medications ---
    expected_meds = len(expected.get('medications', []))
    extracted_meds = len(ext.medications)
    total_expected_meds += expected_meds
    total_extracted_meds += extracted_meds

    # --- Allergies ---
    expected_allergies = set(a.upper() for a in expected.get('allergies', []))
    extracted_allergies = set(a.upper() for a in ext.allergies)
    allergy_match = expected_allergies == extracted_allergies

    # --- Interactions ---
    expected_ints = expected.get('potential_interactions', [])
    extracted_ints = ext.potential_interactions
    total_expected_interactions += len(expected_ints)
    total_extracted_interactions += len(extracted_ints)

    expected_pairs = set()
    for inter in expected_ints:
        pair = tuple(sorted([inter['drug1'].upper(), inter['drug2'].upper()]))
        expected_pairs.add(pair)

    extracted_pairs = set()
    for inter in extracted_ints:
        pair = tuple(sorted([inter.drug1.upper(), inter.drug2.upper()]))
        extracted_pairs.add(pair)

    matched_pairs = expected_pairs & extracted_pairs
    missed_pairs = expected_pairs - extracted_pairs
    extra_pairs = extracted_pairs - expected_pairs

    is_complete = extracted_meds >= expected_meds and allergy_match
    if is_complete:
        complete_notes += 1

    status = "COMPLETE" if is_complete else "PARTIAL"
    print(f"{note_data['note_id']}: meds {extracted_meds}/{expected_meds}, "
          f"allergies {'match' if allergy_match else 'MISMATCH'}, "
          f"interactions {len(matched_pairs)}/{len(expected_pairs)} matched — {status}")
    if missed_pairs:
        print(f"  Missed: {[f'{a}+{b}' for a,b in missed_pairs]}")
    if extra_pairs:
        print(f"  Extra:  {[f'{a}+{b}' for a,b in extra_pairs]}")

print(f"\n--- Overall ---")
print(f"Total medications: {total_extracted_meds} extracted / {total_expected_meds} expected")
print(f"Total interactions: {total_extracted_interactions} extracted / {total_expected_interactions} expected")
print(f"Complete extractions: {complete_notes}/{len(notes)}")

---
# Part 3: Text-to-SQL — Pharmacist Natural Language Interface (25%)

### Cell 3.1 — Schema Description

In [ ]:
# === CELL 3.1: SCHEMA DESCRIPTION ===

schema_description = """
Table: vista_pharmacy
Key columns:
- MRN (text): encrypted patient medical record number
- DRUG (text): full drug name with strength, e.g. 'TAMOXIFEN 20MG TAB', 'ENOXAPARIN 40MG INJ'
- PHARMACY_ORDERABLE_ITEM (text): generic drug name, e.g. 'TAMOXIFEN', 'ENOXAPARIN'
- QTY (integer): quantity dispensed
- DAYS_SUPPLY (integer): days the prescription covers
- Number_OF_REFILLS (integer): number of refills authorized
- ROUTE (text): 'ORAL', 'TOPICAL', 'SUBCUTANEOUS'
- SCHEDULE (text): dosing schedule, e.g. 'QDAY', 'BID', 'Q8H', 'Q6H PRN'
- STATUS (text): 'ACTIVE', 'EXPIRED', 'DISCONTINUED', 'DISCONTINUED (EDIT)'
- CLINIC (text): prescribing clinic, e.g. 'BREAST ONC-GHADEER', 'KHCC-ICU-CRITICAL CARE'
- PROVIDER (text): prescribing provider name
- DIVISION (text): pharmacy division
- DOSAGE_ORDERED (text): dose as text
- ISSUE_DATE (text): prescription issue date, format 'YYYY-MM-DDTHH:MM:SS.0000000'
- FILL_DATE (text): when filled
- EXPIRATION_DATE (text): when prescription expires
- VERB (text): 'TAKE', 'APPLY', 'INJECT'
- PATIENT_INSTRUCTIONS (text): English instructions or NULL
- OTHER_LANGUAGE_DOSAGE (text): Arabic dosage text

Table: vista_patients
Key columns:
- MRN (text): patient medical record number (JOIN key with vista_pharmacy)
- DOB (text): date of birth
- SEX (text): 'MALE' or 'FEMALE'
- NATIONALITY (text): 'JOR', 'PSE', 'IRQ', 'SYR', etc.
- PROVINCE (text): 'AMMAN', 'IRBID', 'ZARQA', etc.
- MARITAL_STATUS (text): 'MARRIED', 'SINGLE', 'WIDOWED', 'DIVORCED'
- DATE_OF_DEATH (text): '-' if alive, date if deceased

Relationship: vista_pharmacy.MRN = vista_patients.MRN (many pharmacy orders per patient)

Important notes:
- Many pharmacy columns are NULL (billing, NDC, military fields).
- Use PHARMACY_ORDERABLE_ITEM for generic drug name matching.
- Use DRUG for full name with strength.
- STATUS indicates whether an order is current (ACTIVE) or past.
- Use JOINs when questions involve patient demographics.
- Dates are in ISO format as text strings.
"""

sample_pharm = pd.read_sql(
    "SELECT DRUG, QTY, DAYS_SUPPLY, ROUTE, SCHEDULE, STATUS, CLINIC FROM vista_pharmacy LIMIT 3", conn
)
sample_pts = pd.read_sql(
    "SELECT MRN, DOB, SEX, NATIONALITY, PROVINCE FROM vista_patients LIMIT 3", conn
)
print("Schema description created.")
print("\nSample pharmacy rows:")
print(sample_pharm.to_string())
print("\nSample patient rows:")
print(sample_pts.to_string())

### Cell 3.2 — Text-to-SQL Function

In [ ]:
# === CELL 3.2: TEXT TO SQL FUNCTION ===

def text_to_sql(question):
    """Convert a natural language question to a SQL query using few-shot examples."""
    system_prompt = f"""You are a SQL expert for a hospital pharmacy database.
Generate a SQLite SELECT query to answer the user's question.

{schema_description}

Rules:
- Return ONLY the SQL query, nothing else. No explanation, no markdown, no code fences.
- Only generate SELECT queries. Never INSERT, UPDATE, DELETE, DROP, or ALTER.
- Use single quotes for string literals.
- Use PHARMACY_ORDERABLE_ITEM for matching drug names (generic).
- Use JOINs when questions involve patient demographics (sex, nationality, province, age).

Examples:

Question: How many pharmacy orders are there?
SQL: SELECT COUNT(*) as total_orders FROM vista_pharmacy

Question: What are the top 5 most prescribed drugs?
SQL: SELECT PHARMACY_ORDERABLE_ITEM, COUNT(*) as rx_count FROM vista_pharmacy GROUP BY PHARMACY_ORDERABLE_ITEM ORDER BY rx_count DESC LIMIT 5

Question: How many prescriptions do female patients have?
SQL: SELECT COUNT(*) as rx_count FROM vista_pharmacy rx JOIN vista_patients p ON rx.MRN = p.MRN WHERE p.SEX = 'FEMALE'
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        temperature=0
    )

    sql = response.choices[0].message.content.strip()
    sql = sql.replace('```sql', '').replace('```', '').strip()
    return sql

test_sql = text_to_sql("How many total pharmacy orders are in the database?")
print(f"Generated SQL: {test_sql}")

### Cell 3.3 — Full Pipeline

In [ ]:
# === CELL 3.3: FULL PIPELINE ===

def ask_pharmacy(question):
    """Full text-to-SQL pipeline for pharmacy queries."""

    sql = text_to_sql(question)

    sql_upper = sql.strip().upper()
    if not sql_upper.startswith('SELECT'):
        return {'sql': sql, 'answer': 'BLOCKED: Only SELECT queries allowed.', 'data': None}

    dangerous = ['DROP', 'DELETE', 'INSERT', 'UPDATE', 'ALTER', 'CREATE', 'TRUNCATE']
    for kw in dangerous:
        if kw in sql_upper:
            return {'sql': sql, 'answer': f'BLOCKED: Dangerous keyword {kw}.', 'data': None}

    try:
        result_df = pd.read_sql(sql, conn)
    except Exception as e:
        return {'sql': sql, 'answer': f'SQL error: {str(e)}', 'data': None}

    answer_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful clinical pharmacy analyst. Given a question and SQL query results, provide a clear, concise natural language answer. Be specific with numbers."},
            {"role": "user", "content": f"Question: {question}\n\nSQL: {sql}\n\nResults:\n{result_df.to_string()}"}
        ],
        temperature=0
    )

    return {
        'sql': sql,
        'answer': answer_response.choices[0].message.content,
        'data': result_df
    }

print("ask_pharmacy() ready.")

### Cell 3.4 — Test with 5 Pharmacy Questions

In [ ]:
# === CELL 3.4: TEST QUESTIONS ===

questions = [
    "Which patients are on Tamoxifen?",
    "What is the most prescribed drug in the ICU?",
    "Show active prescriptions for patients from Amman",
    "How many oral vs injectable medications are there?",
    "Find patients with more than 5 pharmacy orders"
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    result = ask_pharmacy(q)
    print(f"SQL: {result['sql']}")
    print(f"Answer: {result['answer']}")

---
# Part 4: Critical Analysis (15%)

### 1. Drug Interaction Blind Spots

The LLM identifies interactions based on general pharmacological knowledge encoded in its training data, but it lacks access to patient-specific context that real clinical decision support systems (CDSS) use. For example, it cannot check a patient's current creatinine clearance to adjust doses of nephrotoxic drugs like cisplatin, nor can it detect cumulative toxicity from a patient's prior treatment history. Dose-dependent interactions (e.g., methotrexate at low immunosuppressive doses vs high oncology doses) may be flagged identically regardless of the actual dose. A real system like Lexicomp or First Databank cross-references the specific patient's full medication list in real time, whereas our extraction processes each note independently.

### 2. Text-to-SQL Risks

A plausible but incorrect SQL query is dangerous because the pharmacist may trust the natural language answer without verifying the underlying query. For example, the model might generate a query that filters on DRUG instead of PHARMACY_ORDERABLE_ITEM, returning partial results that appear complete. In a clinical setting, every generated query should be displayed alongside the answer, and critical queries (e.g., allergy checks before dispensing) should require pharmacist confirmation. Automated testing with a known set of question-answer pairs would help catch systematic errors before deployment.

### 3. PHI / Security

Our pipeline sends patient MRNs (even though encrypted) and clinical note text to OpenAI's API, which raises data residency and privacy concerns under Jordanian health regulations and institutional policy. The API key is managed through Colab Secrets, which prevents it from being committed to GitHub, but a production system would need a proper secrets manager (Azure Key Vault, etc.). At KHCC, deployment would require a Business Associate Agreement (or equivalent) with OpenAI, data anonymization before API calls, and potentially an on-premise LLM to keep PHI within the hospital network.

### 4. Missing Data

The synthetic pharmacy data has NULL values in most billing, NDC, allergy ingredient, and insurance columns. A real system would need populated NDC codes for drug identification and formulary checking, DRUG_ALLERGY_INGREDIENTS for automated allergy cross-checking before dispensing, and CHECKING_PHARMACIST for the verification audit trail. The ICD_DIAGNOSIS field is also empty, which means we cannot validate that the prescribed drug is appropriate for the patient's diagnosis — a core pharmacy clinical function.

---
# Part 5: Stretch Challenge (10%)

**Option A: Tool Calling Pharmacist Assistant**

In [ ]:
# === PART 5: TOOL CALLING PHARMACIST ASSISTANT ===

tools = [
    {
        "type": "function",
        "function": {
            "name": "run_sql_query",
            "description": "Execute a SQL SELECT query against the KHCC pharmacy database containing vista_pharmacy and vista_patients tables.",
            "parameters": {
                "type": "object",
                "properties": {
                    "sql": {"type": "string", "description": "The SQL SELECT query to execute."}
                },
                "required": ["sql"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_drug_interaction",
            "description": "Check if two drugs have a known interaction and return the risk assessment.",
            "parameters": {
                "type": "object",
                "properties": {
                    "drug1": {"type": "string", "description": "First drug name."},
                    "drug2": {"type": "string", "description": "Second drug name."}
                },
                "required": ["drug1", "drug2"]
            }
        }
    }
]


def run_sql_query(sql):
    sql_upper = sql.strip().upper()
    if not sql_upper.startswith('SELECT'):
        return "ERROR: Only SELECT queries allowed."
    try:
        result_df = pd.read_sql(sql, conn)
        return result_df.to_string()
    except Exception as e:
        return f"ERROR: {str(e)}"


KNOWN_INTERACTIONS = {
    ("TAMOXIFEN", "ANASTROZOLE"): ("high", "Concurrent use is contraindicated — opposing mechanisms of action."),
    ("DOXORUBICIN", "PACLITAXEL"): ("high", "Sequential use increases cardiotoxicity risk; paclitaxel can elevate doxorubicin plasma levels."),
    ("DOXORUBICIN", "BLEOMYCIN"): ("high", "Additive pulmonary and cardiotoxicity; requires cumulative dose tracking and PFT monitoring."),
    ("FLUCONAZOLE", "LEVOFLOXACIN"): ("high", "Both prolong QT interval; concurrent use increases risk of torsades de pointes."),
    ("MORPHINE", "GABAPENTIN"): ("high", "Combined CNS depression — increased risk of respiratory depression."),
    ("FENTANYL", "GABAPENTIN"): ("high", "Combined CNS depression — FDA boxed warning for respiratory depression."),
    ("6-MERCAPTOPURINE", "METHOTREXATE"): ("high", "Methotrexate inhibits xanthine oxidase, increasing 6-MP levels and toxicity."),
    ("DOXORUBICIN", "CYCLOPHOSPHAMIDE"): ("moderate", "Additive cardiotoxicity and myelosuppression; cumulative dose monitoring required."),
    ("CISPLATIN", "ETOPOSIDE"): ("moderate", "Additive myelosuppression and nephrotoxicity; requires renal function monitoring."),
    ("OMEPRAZOLE", "LEVOTHYROXINE"): ("moderate", "PPIs reduce gastric acid, decreasing levothyroxine absorption; separate dosing and monitor TSH."),
    ("BEVACIZUMAB", "IRINOTECAN"): ("moderate", "Bevacizumab increases risk of GI perforation when combined with irinotecan."),
    ("CELECOXIB", "ENOXAPARIN"): ("moderate", "NSAIDs increase bleeding risk with anticoagulants."),
    ("DEXAMETHASONE", "METFORMIN"): ("moderate", "Corticosteroids can increase blood glucose, reducing metformin efficacy."),
    ("IMATINIB", "ESOMEPRAZOLE"): ("moderate", "Proton pump inhibitors may reduce imatinib absorption."),
    ("CAPECITABINE", "METFORMIN"): ("moderate", "Capecitabine may increase metformin toxicity via renal impairment."),
    ("METOCLOPRAMIDE", "PARACETAMOL"): ("low", "Metoclopramide accelerates gastric emptying and increases paracetamol absorption rate."),
}


def check_drug_interaction(drug1, drug2):
    key1 = (drug1.upper(), drug2.upper())
    key2 = (drug2.upper(), drug1.upper())
    if key1 in KNOWN_INTERACTIONS:
        level, desc = KNOWN_INTERACTIONS[key1]
        return f"INTERACTION FOUND [{level.upper()}]: {drug1} + {drug2} — {desc}"
    elif key2 in KNOWN_INTERACTIONS:
        level, desc = KNOWN_INTERACTIONS[key2]
        return f"INTERACTION FOUND [{level.upper()}]: {drug1} + {drug2} — {desc}"
    else:
        return f"No known interaction between {drug1} and {drug2} in our database."


def pharmacist_assistant(question):
    """Multi-turn pharmacist assistant with tool calling."""
    messages = [
        {
            "role": "system",
            "content": f"""You are a clinical pharmacy assistant at KHCC.
Use the available tools to answer pharmacy questions.
- Use run_sql_query to look up medication orders and patient data.
- Use check_drug_interaction to verify drug-drug interactions.

{schema_description}

Always use the tools rather than guessing. Provide clear, clinical answers."""
        },
        {"role": "user", "content": question}
    ]

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
        temperature=0
    )

    assistant_msg = response.choices[0].message

    if assistant_msg.tool_calls:
        messages.append(assistant_msg)

        for tool_call in assistant_msg.tool_calls:
            args = json.loads(tool_call.function.arguments)
            func_name = tool_call.function.name
            print(f"  Tool called: {func_name}({args})")

            if func_name == "run_sql_query":
                result = run_sql_query(args['sql'])
            elif func_name == "check_drug_interaction":
                result = check_drug_interaction(args['drug1'], args['drug2'])
            else:
                result = "Unknown tool."

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })

        final = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0
        )
        return final.choices[0].message.content
    else:
        return assistant_msg.content


# Test the assistant
test_questions = [
    "What medications is the oldest patient in the database taking?",
    "Is there an interaction between Tamoxifen and Anastrozole?",
    "Check if any patients are on both Morphine and Gabapentin — is that safe?",
]

for q in test_questions:
    print(f"\n{'='*60}")
    print(f"Pharmacist: {q}")
    answer = pharmacist_assistant(q)
    print(f"Assistant: {answer}")

---
## Submission Checklist

- [x] Part 1: Pharmacy data loaded into SQLite with 5 SQL queries and 3 JOIN queries
- [x] Part 2: Pydantic models defined, all 10 notes extracted, cross-referenced with pharmacy DB
- [x] Part 3: text_to_sql and ask_pharmacy functions working, 5 test questions answered
- [x] Part 4: Critical analysis (200-400 words) referencing this implementation
- [x] Part 5: Stretch challenge completed (Option A — Tool Calling)
- [x] Notebook runs top-to-bottom without errors
- [ ] Pushed to GitHub with 3+ descriptive commits
- [x] No real patient data in the notebook